# 05 — ATES classifier

**Time:** 15:05–15:45

This notebook runs `atesCore` — the classifier `runAutoATES` actually calls.
Infrequent PRA/runout set the outer envelope; typical drives inner classes.
The floors-then-average hybrid in the v3.0 paper lives in `atesValidation`
and is the research path, not this live call.

Open the result next to the expert consensus overlay. Discuss, do not score
it as an exam. Inter-mapper agreement at Connaught is the ceiling.

Class 0 is a safety decision: calling avalanche terrain non-avalanche is the
error that hurts someone.


In [ ]:
from pathlib import Path
import os, sys
sys.path.insert(0, str(Path(".").resolve()))
from workshop import REPO, DATA, OUT, need, read_raster, show_ates
import matplotlib.pyplot as plt

os.chdir(REPO)
dem = need(OUT / "PRA" / "autoATES_v3_DSM_00.tif",
           DATA / "01_elevation" / "outputs_reference" / "dem_aligned.tif")
forest = need(OUT / "PRA" / "autoATES_v3_FOREST_CANOPY_00.tif",
              DATA / "02_forest" / "outputs_reference" / "sen2_canopy_cover_aligned.tif")

In [ ]:
from autoates.comAutoATES.comAutoATES import load_config
from autoates.comAutoATES.ates.atesCore import run_ates_classification

cfg = load_config(REPO / "config" / "autoATESCfg_workshop.ini")

def latest(scen, token):
    hits = list((OUT / "FlowPy" / scen).glob(f"res_*/com4_*_run_{token}.tif"))
    if hits:
        return sorted(hits)[-1]
    hits = list((DATA / "04_runout" / "outputs_reference" / scen).glob(f"*_{token}.tif"))
    return sorted(hits)[-1]

runout_by_scenario, pra_by_scenario = {}, {}
for scen in ("frequent", "extreme"):
    runout_by_scenario[scen] = {
        "zDelta": latest(scen, "zdelta"),
        "routFluxSum": latest(scen, "routFluxSum"),
        "fpTravelAngleMax": latest(scen, "fpTravelAngleMax"),
    }
    pra_by_scenario[scen] = {
        "sieve": need(OUT / "PRA" / f"pra_{scen}_pra_sieve.tif",
                      DATA / "03_pra" / "outputs_reference" / f"pra_{scen}_pra_sieve.tif"),
        "continuous": need(OUT / "PRA" / f"pra_{scen}_pra_continuous.tif",
                           DATA / "03_pra" / "outputs_reference" / f"pra_{scen}_pra_continuous.tif"),
    }

ates_dir = OUT / "ATES"
out = run_ates_classification(
    dem_path=dem, forest_path=forest,
    runout_by_scenario=runout_by_scenario, pra_by_scenario=pra_by_scenario,
    output_dir=ates_dir, config=cfg,
    aoi_path=Path(cfg["General"]["aoi_path"]),
)
print("ATES written:", out)

In [ ]:
arr, *_ = read_raster(ates_dir / "ATES_classification.tif")
fig, ax = plt.subplots(figsize=(7, 7))
show_ates(arr, "autoATES v3.0 — Connaught Creek", ax=ax)
plt.tight_layout()
print("QGIS style: qgis/ates_classes.qml")
print("Expert overlay: data/00_orientation/inputs/expert_consensus.shp")

**If stuck:** `data/05_ates/outputs_reference/ATES_classification.tif` plus
`qgis/ates_classes.qml`.
